# dextra — اختبار شامل لكل دوال المكتبة
هذا الدفتر يستعرض ويختبر **كل دالة عامة** في مكتبة dextra من Phase 1 حتى Phase 6.

- كل قسم يخص مرحلة (phase) ويستدعي دوالها باستدعاء صحيح مع بيانات اصطناعية.
- الخلية الأخيرة **فحص آلي شامل** يستدعي كل الدوال بـ `show=False, plot=False` ويطبع جدول PASS/FAIL.

**التشغيل:** فعّل البيئة ثم `Run All`. يتطلب `pip install -e ".[dev,ml]"` (scikit-learn + scipy + plotly).

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
import dextra as dx
print('dextra', dx.__version__)
print('public functions:', sum(1 for n in dx.__all__ if callable(getattr(dx, n, None))))

## 0. البيانات الاصطناعية
نبني مجموعتي بيانات: `df` نظيفة غنية، و`df_messy` متّسخة عمداً لاختبار التنظيف، و`df_nan` بقيم مفقودة.

In [ ]:
rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    'age': rng.integers(18, 70, n),
    'income': rng.lognormal(10, 0.5, n),          # موزّعة ملتوية
    'score': rng.normal(50, 12, n),
    'spend': rng.normal(200, 40, n),
    'city': rng.choice(['Cairo','Giza','Alex'], n),
    'gender': rng.choice(['M','F'], n),
    'signup': pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0,500,n), unit='D'),
})
df['price'] = 3*df['age'] + 0.0008*df['income'] + 2*df['score'] + rng.normal(0,10,n)   # هدف انحدار
_lin = (df['score']-50)/12 + (df['spend']-200)/40 + rng.normal(0,0.5,n)
df['churn'] = np.where(_lin > 0, 'yes', 'no')                                          # هدف تصنيف

# مجموعة بقيم مفقودة
df_nan = df.copy()
for c in ['income','score','city']:
    idx = rng.choice(n, size=15, replace=False)
    df_nan.loc[idx, c] = np.nan

# مجموعة متّسخة: أسماء أعمدة فوضوية + تكرارات + قيم مفقودة + شواذ
df_messy = pd.DataFrame({
    ' First Name ': ['  Ann','Bob','Bob','CARLA',None,'Dan','Dan'],
    'AGE': ['25','30','30','40','45','50','50'],
    'Income($)': [5000, 6000, 6000, 7000, np.nan, 999999, 8000],   # شاذّ + مفقود
    'City': ['Cairo','Giza','Giza','Alex','Cairo','Cairo','Cairo'],
})
# الأعمدة الرقمية المستخدمة كثيراً
NUM = ['age','income','score','spend']
df.head()

## Phase 1 — الإحصاء الوصفي والرسوم الأساسية
`describe_numeric`, `plot_histograms`, `plot_boxplots`

In [ ]:
dx.describe_numeric(df)

In [ ]:
dx.plot_histograms(df, cols=['age','income','score'])

In [ ]:
dx.plot_boxplots(df, cols=['age','score','spend'])

## Phase 2 — الإحصاء المتقدّم (22 دالة)
وصفية موسّعة، ثنائية المتغيّر، أدوات EDA، استدلال، اختبارات فرضيات، تشخيص ML.

In [ ]:
dx.z_scores(df, cols=['age','score'])

In [ ]:
dx.pearson_skewness(df, cols=['income','score'])

In [ ]:
dx.empirical_rule_check(df, cols=['score'])

In [ ]:
dx.outliers_report(df, cols=['income'])

In [ ]:
dx.correlation_matrix(df, cols=['age','income','score','spend','price'])

In [ ]:
dx.simple_linear_regression(df, x='score', y='price')

In [ ]:
dx.missing_report(df_nan)

In [ ]:
dx.frequency_table(df, col='city')

In [ ]:
dx.cross_tab(df, row='city', col='gender')

In [ ]:
dx.group_compare(df, group_col='city', value_cols=['price','score'])

In [ ]:
dx.confidence_interval_mean(df['score'])

In [ ]:
dx.confidence_interval_proportion(successes=int((df['churn']=='yes').sum()), n=len(df))

In [ ]:
dx.sample_size_mean(margin_error=2.0, std=12.0)

In [ ]:
dx.sample_size_proportion(margin_error=0.05)

In [ ]:
dx.normality_test(df['score'])

In [ ]:
dx.t_test_one_sample(df['score'], popmean=50)

In [ ]:
dx.t_test_two_sample(df.loc[df.gender=='M','score'], df.loc[df.gender=='F','score'])

In [ ]:
# قياسات مزدوجة (قبل/بعد)
before = df['score'].to_numpy()
after = before + rng.normal(1.5, 3, len(before))
dx.t_test_paired(before, after)

In [ ]:
dx.anova_oneway(df, group_col='city', value_col='price')

In [ ]:
dx.chi_square_independence(df, row='city', col='gender')

In [ ]:
dx.vif_scores(df, cols=NUM)

In [ ]:
dx.class_imbalance(df['churn'])

## Phase 3 — التنظيف (10 دوال)
مفتّشون (`*_show`) لا يعدّلون البيانات، ومُصلِحون يعيدون نسخة منظّفة، مع تقرير جودة وقواعد عمل.

In [ ]:
dx.clean_report(df_messy)

In [ ]:
dx.na_show(df_nan)

In [ ]:
dx.dup_show(df_messy)

In [ ]:
dx.out_show(df, cols=['income'])

In [ ]:
dx.standardize_columns(df_messy)

In [ ]:
dx.cast_types(df_messy)

In [ ]:
rules = [
    {'name': 'age_valid',     'check': 'age.between(0, 120)'},
    {'name': 'price_positive','check': 'price >= 0'},
]
dx.validate_rules(df, rules)

In [ ]:
dx.handle_missing(df_nan)

In [ ]:
dx.dedupe(df_messy)

In [ ]:
dx.clip_outliers(df, cols=['income'])

## Phase 4 — هندسة الميزات (8 دوال)
إطار fit/apply آمن ضد التسرّب: نتعلّم على `df_tr` ونطبّق حرفياً على `df_te`.

In [ ]:
df_tr = df.iloc[:150].copy()
df_te = df.iloc[150:].copy()
print('train', df_tr.shape, '| test', df_te.shape)

In [ ]:
# transform — fit ثم apply
_, p = dx.transform(df_tr, cols=['income'], method='log1p', return_params=True)
dx.transform(df_te, params=p, show=True, plot=False)
dx.transform(df, cols=['income'], method='compare')

In [ ]:
_, p = dx.scale(df_tr, cols=['age','score'], method='standard', return_params=True)
dx.scale(df_te, params=p, show=True, plot=False)

In [ ]:
_, p = dx.bin(df_tr, cols=['price'], method='quantile', n_bins=4, return_params=True)
dx.bin(df_te, params=p, show=True, plot=False)

In [ ]:
_, p = dx.encode(df_tr, cols=['city'], method='onehot', return_params=True)
dx.encode(df_te, params=p, show=True, plot=False)

In [ ]:
_, p = dx.dtfeats(df_tr, cols=['signup'], method='both', return_params=True)
dx.dtfeats(df_te, params=p, show=True, plot=False)

In [ ]:
_, p = dx.cross(df_tr, pairs=[('score','spend')], method='product', return_params=True)
dx.cross(df_te, params=p, show=True, plot=False)

In [ ]:
_, p = dx.aggfeat(df_tr, group='city', value='price', agg='mean', return_params=True)
dx.aggfeat(df_te, params=p, show=True, plot=False)

In [ ]:
# featpipe — سلسلة من الخطوات
recipe = [
    {'fn': 'transform', 'cols': ['income'], 'method': 'log1p'},
    {'fn': 'scale', 'cols': ['age','score'], 'method': 'robust'},
    {'fn': 'encode', 'cols': ['city'], 'method': 'onehot'},
]
df_feat, p = dx.featpipe(df_tr, steps=recipe, return_params=True)
df_feat_te = dx.featpipe(df_te, params=p)   # apply
df_feat.head()

## Phase 5 — اختيار الميزات (5 دوال)
عائلات Filter / Embedded / Wrapper، إطار fit/apply.

In [ ]:
dx.redundancy(df_tr, cols=NUM, method='correlation', threshold=0.9)

In [ ]:
dx.relevance(df_tr, y='churn', cols=NUM, method='anova', keep=3)

In [ ]:
dx.importance(df_tr, y='churn', cols=NUM, method='tree', keep=3)

In [ ]:
dx.rfe(df_tr, y='churn', cols=NUM, keep=2, estimator='tree')

In [ ]:
recipe_sel = [
    {'fn': 'redundancy', 'method': 'correlation', 'threshold': 0.95},
    {'fn': 'relevance', 'method': 'anova', 'keep': 4},
    {'fn': 'importance', 'method': 'tree', 'keep': 3},
]
dx.selectpipe(df_tr[NUM + ['churn']], steps=recipe_sel, y='churn')

## Phase 6 — النمذجة (3 دوال)
خطوط أساس فورية بنمط fit/apply/compare + artifact هجين.

In [ ]:
# regress — انحدار
_, p = dx.regress(df_tr, y='price', cols=NUM, method='forest', return_params=True)
dx.regress(df_te, params=p, show=True, plot=False)        # apply
dx.regress(df_tr, y='price', cols=NUM, method='compare')  # مقارنة

In [ ]:
# classify — تصنيف
_, p = dx.classify(df_tr, y='churn', cols=NUM, method='forest', return_params=True)
dx.classify(df_te, params=p, show=True, plot=False)        # apply
dx.classify(df_tr, y='churn', cols=NUM, method='compare')  # مقارنة

In [ ]:
# cluster — تجميع (بلا هدف)
_, p = dx.cluster(df, cols=NUM, method='kmeans', return_params=True)  # اختيار k تلقائي
dx.cluster(df, cols=NUM, method='compare')                            # مقارنة

## ✅ فحص آلي شامل لكل الدوال
يستدعي كل دالة عامة بصمت (`show=False, plot=False`) ويتحقّق أنها تعمل دون أخطاء.

In [ ]:
matplotlib.use('Agg', force=True)
KW = dict(show=False, plot=False)
_, P_FEAT = dx.transform(df_tr, cols=['income'], method='log1p', return_params=True, **KW)
_, P_REG = dx.regress(df_tr, y='price', cols=NUM, method='forest', return_params=True, **KW)
_, P_CLF = dx.classify(df_tr, y='churn', cols=NUM, method='forest', return_params=True, **KW)
_, P_CLUS = dx.cluster(df, cols=NUM, method='kmeans', return_params=True, **KW)

CALLS = {
  'describe_numeric': lambda: dx.describe_numeric(df, show=False),
  'plot_histograms':  lambda: dx.plot_histograms(df, cols=['age'], show=False, return_fig=True),
  'plot_boxplots':    lambda: dx.plot_boxplots(df, cols=['age'], show=False, return_fig=True),
  'z_scores':         lambda: dx.z_scores(df, cols=['age'], **KW),
  'pearson_skewness': lambda: dx.pearson_skewness(df, cols=['income'], **KW),
  'empirical_rule_check': lambda: dx.empirical_rule_check(df, cols=['score'], **KW),
  'outliers_report':  lambda: dx.outliers_report(df, cols=['income'], **KW),
  'correlation_matrix': lambda: dx.correlation_matrix(df, cols=NUM, **KW),
  'simple_linear_regression': lambda: dx.simple_linear_regression(df, x='score', y='price', **KW),
  'missing_report':   lambda: dx.missing_report(df_nan, **KW),
  'frequency_table':  lambda: dx.frequency_table(df, col='city', **KW),
  'cross_tab':        lambda: dx.cross_tab(df, row='city', col='gender', **KW),
  'group_compare':    lambda: dx.group_compare(df, group_col='city', value_cols=['price'], **KW),
  'confidence_interval_mean': lambda: dx.confidence_interval_mean(df['score'], **KW),
  'confidence_interval_proportion': lambda: dx.confidence_interval_proportion(40, 200, **KW),
  'sample_size_mean': lambda: dx.sample_size_mean(2.0, 12.0, **KW),
  'sample_size_proportion': lambda: dx.sample_size_proportion(0.05, **KW),
  'normality_test':   lambda: dx.normality_test(df['score'], **KW),
  't_test_one_sample':lambda: dx.t_test_one_sample(df['score'], popmean=50, **KW),
  't_test_two_sample':lambda: dx.t_test_two_sample(df.loc[df.gender=='M','score'], df.loc[df.gender=='F','score'], **KW),
  't_test_paired':    lambda: dx.t_test_paired(before, after, **KW),
  'anova_oneway':     lambda: dx.anova_oneway(df, group_col='city', value_col='price', **KW),
  'chi_square_independence': lambda: dx.chi_square_independence(df, row='city', col='gender', **KW),
  'vif_scores':       lambda: dx.vif_scores(df, cols=NUM, **KW),
  'class_imbalance':  lambda: dx.class_imbalance(df['churn'], **KW),
  'clean_report':     lambda: dx.clean_report(df_messy, **KW),
  'standardize_columns': lambda: dx.standardize_columns(df_messy, **KW),
  'cast_types':       lambda: dx.cast_types(df_messy, **KW),
  'validate_rules':   lambda: dx.validate_rules(df, [{'name':'age_valid','check':'age.between(0,120)'}], **KW),
  'handle_missing':   lambda: dx.handle_missing(df_nan, **KW),
  'dedupe':           lambda: dx.dedupe(df_messy, **KW),
  'clip_outliers':    lambda: dx.clip_outliers(df, cols=['income'], **KW),
  'na_show':          lambda: dx.na_show(df_nan, **KW),
  'dup_show':         lambda: dx.dup_show(df_messy, **KW),
  'out_show':         lambda: dx.out_show(df, cols=['income'], **KW),
  'transform':        lambda: dx.transform(df_tr, cols=['income'], method='log1p', **KW),
  'scale':            lambda: dx.scale(df_tr, cols=['age'], method='standard', **KW),
  'bin':              lambda: dx.bin(df_tr, cols=['price'], method='quantile', n_bins=4, **KW),
  'encode':           lambda: dx.encode(df_tr, cols=['city'], method='onehot', **KW),
  'dtfeats':          lambda: dx.dtfeats(df_tr, cols=['signup'], method='both', **KW),
  'cross':            lambda: dx.cross(df_tr, pairs=[('score','spend')], method='product', **KW),
  'aggfeat':          lambda: dx.aggfeat(df_tr, group='city', value='price', agg='mean', **KW),
  'featpipe':         lambda: dx.featpipe(df_tr, steps=[{'fn':'scale','cols':['age'],'method':'standard'}], **KW),
  'redundancy':       lambda: dx.redundancy(df_tr, cols=NUM, method='correlation', threshold=0.9, **KW),
  'relevance':        lambda: dx.relevance(df_tr, y='churn', cols=NUM, method='anova', keep=3, **KW),
  'importance':       lambda: dx.importance(df_tr, y='churn', cols=NUM, method='tree', keep=3, **KW),
  'rfe':              lambda: dx.rfe(df_tr, y='churn', cols=NUM, keep=2, estimator='tree', **KW),
  'selectpipe':       lambda: dx.selectpipe(df_tr[NUM+['churn']], steps=[{'fn':'relevance','method':'anova','keep':3}], y='churn', **KW),
  'regress':          lambda: dx.regress(df_tr, y='price', cols=NUM, method='forest', **KW),
  'regress(apply)':   lambda: dx.regress(df_te, params=P_REG, **KW),
  'classify':         lambda: dx.classify(df_tr, y='churn', cols=NUM, method='forest', **KW),
  'classify(apply)':  lambda: dx.classify(df_te, params=P_CLF, **KW),
  'cluster':          lambda: dx.cluster(df, cols=NUM, method='kmeans', **KW),
  'cluster(apply)':   lambda: dx.cluster(df, params=P_CLUS, **KW),
}

rows, passed = [], 0
for name, fn in CALLS.items():
    try:
        fn(); rows.append((name, 'PASS', '')); passed += 1
    except Exception as e:
        rows.append((name, 'FAIL', f'{type(e).__name__}: {e}'))

res = pd.DataFrame(rows, columns=['function','status','error'])
print(f'{passed}/{len(CALLS)} calls passed')
res